In [13]:
!pip install pyfaidx transformers datasets tqdm
!pip install transformers accelerate --quiet

import pandas as pd
import requests
from tqdm import tqdm
from pyfaidx import Fasta
from transformers import BertTokenizer, BertForSequenceClassification
import torch

Defaulting to user installation because normal site-packages is not writeable


In [31]:
# ========================
#  📌 第二步：读取 VCF 文件（你需要在 Colab 手动上传）
# ========================
vcf_filename = "ClinVar_NonCoding_SNV_PB.vcf"  # 你需要替换为自己的 VCF 文件名

''' # 解析 VCF 文件
vcf_data = []
with open(vcf_filename, "r") as file:
    for line in file:
        if not line.startswith("#"):  # 跳过注释行
            fields = line.strip().split("\t")
            chrom, pos, ref, alt = fields[0], int(fields[1]), fields[3], fields[4]
            vcf_data.append([chrom, pos, ref, alt]) '''

# 解析 VCF 文件，同时提取 `INFO` 字段
vcf_data = []
with open(vcf_filename, "r") as file:
    for line in file:
        if not line.startswith("#"):  # 跳过注释行
            fields = line.strip().split("\t")

            # 提取关键字段
            chrom, pos, ref, alt, info = fields[0], int(fields[1]), fields[3], fields[4], fields[7]

            # 将数据存入列表
            vcf_data.append([chrom, pos, ref, alt, info])

# 创建 DataFrame
df_vcf = pd.DataFrame(vcf_data, columns=["CHROM", "POS", "REF", "ALT", "INFO"])


#df_vcf = pd.DataFrame(vcf_data, columns=["CHROM", "POS", "REF", "ALT"])

# 生成 True_Label（1=致病, 0=良性）
#df_vcf["True_Label"] = df_vcf["INFO"]

print("✅ 解析 VCF 完成！")




✅ 解析 VCF 完成！


In [32]:
df_vcf

,CHROM,POS,REF,ALT,INFO
0,11,126275389,C,T,1.0
1,11,126277517,A,G,1.0
2,6,26093215,G,T,1.0
3,2,19945787,T,C,1.0
4,20,25302322,G,A,1.0
...,...,...,...,...,...
136920,5,75416973,G,A,1.0
136921,X,41346238,C,G,1.0
136922,7,140801551,T,C,1.0
136923,9,121314019,A,G,1.0


In [33]:
# ========================
#  📌 第三步：下载 GRCh38 参考基因组
# ========================
!wget -c http://hgdownload.cse.ucsc.edu/goldenpath/hg38/bigZips/hg38.fa.gz
!gunzip -k hg38.fa.gz

--2025-04-13 06:49:22--  http://hgdownload.cse.ucsc.edu/goldenpath/hg38/bigZips/hg38.fa.gz
128.114.119.163nload.cse.ucsc.edu (hgdownload.cse.ucsc.edu)... 
connected. to hgdownload.cse.ucsc.edu (hgdownload.cse.ucsc.edu)|128.114.119.163|:80... 
416 Requested Range Not Satisfiablee... 

    The file is already fully retrieved; nothing to do.

gzip: hg38.fa already exists; do you wish to overwrite (y or n)? ^C


In [16]:
# 加载参考基因组
genome = Fasta("hg38.fa")

# ========================
#  📌 第四步：提取突变上下游 50bp 序列（共 101bp）
# ========================

def get_sequence(row, flank_size=256):
    try:
        chrom = str(row["#CHROM"])
        if not chrom.startswith("chr"):
            chrom = "chr" + chrom

        pos = int(row["POS"])
        start = max(0, pos - flank_size - 1)
        end = pos + flank_size

        # 染色体是否在 genome 中
        if chrom not in genome:
            return None

        seq = genome[chrom][start:end].seq.upper()

        # 检查长度
        if len(seq) != (2 * flank_size + 1):
            return None

        return seq
    except Exception as e:
        #print(f"[⚠️ get_sequence] Error: {e}")
        return None


def generate_mutant_sequence(row, flank_size=256):
    try:
        seq = list(row["Context_Sequence"])
        mut_pos = flank_size  # 中央是突变位点

        # 检查 REF/ALT 长度是否为单碱基
        if len(row["REF"]) != 1 or len(row["ALT"]) != 1:
            return None

        # 确保 REF 和参考序列匹配
        if seq[mut_pos] != row["REF"]:
            return None

        seq[mut_pos] = row["ALT"]
        return "".join(seq)
    except Exception as e:
        print(f"[⚠️ generate_mutant_sequence] Error: {e}")
        return None


# 设置窗口长度
window = 3000

# 获取上下文序列（参考）
from tqdm import tqdm
tqdm.pandas()
df_vcf["Context_Sequence"] = df_vcf.progress_apply(lambda row: get_sequence(row, flank_size=window), axis=1)

# 删除无效行（可能是 indel 或边界错误）
df_vcf.dropna(subset=["Context_Sequence"], inplace=True)

# 生成突变序列
df_vcf["Mutant_Sequence"] = df_vcf.progress_apply(lambda row: generate_mutant_sequence(row, flank_size=window), axis=1)

# 删除突变失败的行
df_vcf.dropna(subset=["Mutant_Sequence"], inplace=True)

print(f"✅ 成功生成上下文序列和突变序列，共 {len(df_vcf)} 条有效记录")



100%|██████████| 118442/118442 [00:16<00:00, 7114.88it/s]

✅ 成功生成上下文序列和突变序列，共 118441 条有效记录


In [15]:
df_vcf

,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,exon,CDS,start_codon,stop_codon,five_prime_UTR,three_prime_UTR,intron,promoter,other
0,6,26093008,15053,G,A,-,-,0.0,0,0,0,0,0,0,1,0,0
1,2,19989284,15059,T,C,-,-,1.0,0,0,0,0,0,0,1,0,0
2,10,97600167,15071,G,T,-,-,1.0,0,0,0,0,0,0,1,0,0
3,2,63492941,15084,C,A,-,-,1.0,0,0,0,0,0,0,1,0,0
4,1,45014071,15106,G,C,-,-,1.0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118439,15,98939361,3530610,G,A,-,-,1.0,0,0,0,0,0,0,1,0,0
118440,16,2097318,3532850,C,T,-,-,1.0,0,0,0,0,0,0,1,0,0
118441,11,57811488,3533917,T,G,-,-,1.0,0,0,0,0,0,0,1,0,0
118442,1,164560014,3533930,G,A,-,-,1.0,0,0,0,0,0,0,1,0,0


In [14]:
df_vcf = pd.read_csv("ClinVar_NonCoding_SNV_PB.csv", sep="\t")

In [37]:
df_1 = df_vcf

In [51]:
df_vcf = df_1

In [47]:
df_vcf = df_vcf.head(500)

In [17]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
from tqdm import tqdm

# ========================
# ✅ 设置设备（优先使用 GPU）
# ========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔥 Using device: {device}")

# ========================
# ✅ 加载模型和 tokenizer
# ========================
model_name = "InstaDeepAI/nucleotide-transformer-v2-500m-multi-species"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForMaskedLM.from_pretrained(model_name, trust_remote_code=True)
model.to(device)
model.eval()

# ========================
# ✅ 提取 embedding 的函数（平均池化）【支持 GPU】
# ========================
def predict_sequence(sequence):
    inputs = tokenizer(sequence, return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}  # ✅ 移动到 GPU
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        embedding = outputs.hidden_states[-1].mean(dim=1).squeeze().cpu().tolist()  # ✅ 回到 CPU
    return embedding

# ========================
# ✅ Wild-type embedding 提取
# ========================
predictions = []
for _, row in tqdm(df_vcf.iterrows(), total=len(df_vcf), desc="Predicting with NT v2"):
    pred = predict_sequence(row["Context_Sequence"])
    predictions.append(pred)

df_vcf["NT_Predictions"] = predictions

# ========================
# ✅ Mutant embedding 提取
# ========================
predictions_m = []
for _, row in tqdm(df_vcf.iterrows(), total=len(df_vcf), desc="Predicting with NT v2 (Mutant)"):
    pred_m = predict_sequence(row["Mutant_Sequence"])
    predictions_m.append(pred_m)

df_vcf["NT_Predictions_Mutant"] = predictions_m

# ========================
# ✅ 保存
# ========================
df_vcf.to_pickle("nt_predictions_nc.pkl")
print("✅ NT v2 预测完成并保存：nt_predictions.pkl")


🔥 Using device: cuda


IOPub message rate exceeded.████▋     | 54784/118441 [1:36:17<1:51:38,  9.50it/s]
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

Predicting with NT v2 (Mutant): 100%|██████████| 118441/118441 [3:28:12<00:00,  9.48it/s] 


✅ NT v2 预测完成并保存：nt_predictions.pkl


In [18]:
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity

# 将 embedding 转换为矩阵
ref_embeddings = np.array(df_vcf["NT_Predictions"].to_list())
mut_embeddings = np.array(df_vcf["NT_Predictions_Mutant"].to_list())

# 计算余弦相似度（越小越可能致病）
cos_sim = np.array([
    cosine_similarity(ref.reshape(1, -1), mut.reshape(1, -1))[0, 0]
    for ref, mut in zip(ref_embeddings, mut_embeddings)
])

# 使用 1 - cos_sim 作为 deleterious 分数（越大越可能致病）
deleterious_scores = 1 - cos_sim

# 获取真实标签
true_labels = df_vcf["INFO"].values

# 计算 AUC
auc_score = roc_auc_score(true_labels, deleterious_scores)
print(auc_score)


0.9411581949696943
